#Read csv file using data frame reader API

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config



In [0]:
%run ../00-common/02.BronzeHelper

In [0]:
source_path=f"{landing_folder_path}/{v_batch_id}/sprints"
table_name=f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *


sprints_schema = StructType([
  StructField("date", DateType(), True),
  StructField("raceName", StringType(), True),
  StructField("round", IntegerType(), True),
  StructField("season", IntegerType(), True),
  StructField("url", StringType(), True),
  StructField("constructorId", StringType(), True),
  StructField("driverId", StringType(), True),
  StructField("grid", IntegerType(), True),
  StructField("laps", IntegerType(), True),
  StructField("number", IntegerType(), True),
  StructField("points", DoubleType(), True),
  StructField("position", IntegerType(), True),
  StructField("positionText", StringType(), True),
  StructField("status", StringType(), True)
])
#spark.createDataFrame([], driver_schema).printSchema()


#spark.createDataFrame([], driver_schema).printSchema()
df_sprints = (spark.read.format("json")
               .option('header', True)
               .option('mode', 'FAILFAST')  # strict mode datatype validation
               .option('multiLine', True)
               .schema(sprints_schema)
               .load(source_path))

In [0]:
df_sprints.show();

In [0]:
#df_clean = df_sprints.na.drop()
df_clean=df_sprints


In [0]:
import pyspark.sql.functions as F

df_sprints_final=add_ingestion_metadata(df_clean)
display(df_sprints_final)

In [0]:
# (df_sprints_final.write
#       .format("delta")
#       .mode("overwrite")
#       .saveAsTable(table_name))

In [0]:
write_to_bronze(input_df=df_sprints_final,
                target_table=table_name,
                batch_id=v_batch_id)

In [0]:
%sql
select * from formula1_incr_catalog.bronze.sprints;
--select count(*) from formula1_catalog.bronze.results

In [0]:
df_table=spark.read.table(table_name)
display(df_table)

In [0]:
%sql
select season, count(*) from formula1_incr_catalog.bronze.sprints group by season order by season desc